In [ ]:
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

-----


#### ANALISIS LIBLARY
1. Pengolahan Data & File System
os: Digunakan untuk berinteraksi dengan sistem operasi, terutama untuk membaca direktori (folder) dataset dan mendapatkan jalur (path) file gambar.

numpy (np): Pustaka dasar untuk komputasi numerik. Digunakan untuk memanipulasi array gambar, melakukan operasi matematika pada piksel, dan menyimpan data fitur.

pandas (pd): Digunakan untuk mengelola data dalam bentuk tabel (Dataframe). Berguna untuk menyimpan hasil ekstraksi fitur GLCM ke dalam file .csv.

2. Pemrosesan Citra (Image Processing)
cv2 (cv): Pustaka OpenCV. Digunakan sebagai alat utama untuk memuat gambar, mengubah ruang warna (BGR ke Grayscale), melakukan filter (Median Blur), dan operasi morfologi (Morfologi Opening).

skimage.feature (graycomatrix, graycoprops): Pustaka Scikit-Image yang khusus digunakan untuk Ekstraksi Fitur Tekstur GLCM. graycomatrix membuat matriks hubungan antar piksel, dan graycoprops menghitung nilai statistik fitur seperti contrast, correlation, dll.

scipy.stats (entropy): Digunakan untuk menghitung nilai entropy dari matriks GLCM sebagai salah satu fitur tambahan untuk mengukur kompleksitas tekstur citra.

3. Visualisasi Data
matplotlib.pyplot (plt): Pustaka utama untuk menampilkan hasil visual, baik itu gambar pisang, grafik distribusi, maupun plot hasil evaluasi model.

seaborn (sns): Pustaka yang dibangun di atas Matplotlib untuk membuat visualisasi statistik yang lebih menarik, terutama digunakan untuk membuat Heatmap korelasi fitur.

4. Machine Learning & Evaluasi
sklearn.model_selection (train_test_split): Digunakan untuk membagi dataset menjadi dua bagian: data training (untuk melatih model) dan data testing (untuk menguji akurasi model).

sklearn.ensemble (RandomForestClassifier): Algoritma ensemble berbasis pohon keputusan untuk klasifikasi.

sklearn.svm (SVC): Algoritma Support Vector Machine yang efektif untuk klasifikasi data berbasis fitur.

sklearn.neighbors (KNeighborsClassifier): Algoritma K-Nearest Neighbors untuk klasifikasi berbasis kedekatan jarak antar data.

sklearn.metrics: Pustaka untuk mengevaluasi kinerja model.

accuracy_score, classification_report: Mengukur performa akurasi, precision, recall, dan f1-score.

confusion_matrix, ConfusionMatrixDisplay: Digunakan untuk memvisualisasikan matriks kesalahan model (mana kelas yang tertukar).

-----


In [ ]:
data = []
labels = []
file_name = []

for sub_folder in os.listdir("./Banana Ripeness Classification Dataset"):
    sub_folder_path = os.path.join("./Banana Ripeness Classification Dataset", sub_folder)
    
    if os.path.isdir(sub_folder_path):
        sub_folder_files = os.listdir(sub_folder_path)
        for i, filename in enumerate(sub_folder_files):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(sub_folder_path, filename)
                img = cv.imread(img_path)
                
                if img is not None:
                    img = img.astype(np.uint8)
                    data.append(img)
                    labels.append(sub_folder)
                    file_name.append(filename)
        
data = np.array(data, dtype=object)
labels = np.array(labels)
print(f"Berhasil memuat {len(data)} gambar dari folder dataset.")

----

Inisialisasi Wadah: * data = [], labels = [], file_name = []: Membuat tiga list kosong untuk menyimpan data gambar, label (nama folder/kelas), dan nama file secara berurutan.

Perulangan Direktori (os.listdir): * Kode melakukan perulangan untuk mengecek setiap folder yang ada di dalam datasetpisang. os.path.isdir memastikan bahwa yang diproses hanyalah folder, bukan file sampah.

Membaca Gambar (cv.imread):

Di dalam setiap sub-folder, program mencari file dengan ekstensi gambar (.png, .jpg, .jpeg).

cv.imread digunakan untuk mengubah file gambar fisik menjadi matriks angka (array) yang dimengerti oleh Python.

img = img.astype(np.uint8): Memastikan format warna piksel adalah unsigned integer 8-bit (0-255), format standar untuk citra digital.

Pengecekan Kualitas (if img is not None): * Ini adalah langkah pengamanan (safety check). Jika ada file yang rusak atau bukan gambar, program tidak akan mengalami error, melainkan hanya akan melewati file tersebut.

Konversi ke Array NumPy:

data = np.array(data, dtype=object): Mengubah list menjadi format Array NumPy.

Penggunaan dtype=object sangat penting di sini karena gambar-gambar dalam dataset mungkin memiliki ukuran (dimensi) yang berbeda-beda. NumPy membutuhkan tipe object agar bisa menampung matriks dengan ukuran yang tidak seragam dalam satu wadah yang sama.

-----


In [ ]:
unique_labels, counts = np.unique(labels, return_counts=True)
plt.figure(figsize=(8, 5))
sns.barplot(x=unique_labels, y=counts, hue=unique_labels, palette='viridis', legend=False)

plt.title('Grafik Distribusi Jumlah Gambar per Kelas', fontsize=14)
plt.xlabel('Label Kelas', fontsize=12)
plt.ylabel('Jumlah Gambar', fontsize=12)
plt.show()

print("Menampilkan 5 sampel gambar dari dataset:")
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
random_indices = np.random.choice(len(data), 5, replace=False)

for i, idx in enumerate(random_indices):
    img_array = np.array(data[idx], dtype=np.uint8)
    
    img_rgb = cv.cvtColor(img_array, cv.COLOR_BGR2RGB) if len(img_array.shape) == 3 else img_array
    
    axes[i].imshow(img_rgb)
    axes[i].set_title(labels[idx])
    axes[i].axis('off')

plt.tight_layout()
plt.show()

-----

1. Grafik Distribusi Kelas
np.unique(labels, return_counts=True): Berfungsi untuk menghitung berapa jumlah gambar yang tersedia untuk setiap kategori/kelas (misalnya: berapa jumlah gambar pisang matang, mentah, dan busuk).

sns.barplot(...): Visualisasi dalam bentuk diagram batang untuk melihat keseimbangan data (class balance).

Mengapa ini penting? Jika jumlah gambar antar kelas tidak seimbang (misal kelas "Matang" punya 500 gambar, tapi "Busuk" hanya 10 gambar), maka model Machine Learning cenderung akan bias dan tidak akurat dalam memprediksi kelas yang jumlahnya sedikit.

2. Visualisasi Sampel Gambar
np.random.choice(...): Mengambil 5 indeks secara acak dari keseluruhan dataset.

cv.cvtColor(..., cv.COLOR_BGR2RGB): OpenCV secara bawaan membaca warna dalam format BGR (Blue-Green-Red), sedangkan Matplotlib menampilkan warna dalam format RGB (Red-Green-Blue). Proses konversi ini dilakukan agar warna gambar yang ditampilkan di layar terlihat natural dan tidak tertukar (misal: warna pisang tidak jadi biru).

Mengapa ini penting? Sebagai langkah verifikasi visual untuk memastikan bahwa gambar yang termuat di dalam list data adalah benar gambar yang diinginkan, tidak rusak (corrupt), dan label-nya sudah sesuai dengan isi gambarnya.

----

In [ ]:
data_augmented = []
labels_augmented = []
print("Data sebelum augmentasi: ", len(data))

----

data_augmented = [], labels_augmented = []: Inisialisasi wadah kosong untuk menyimpan hasil gambar baru yang telah dimanipulasi.

print("Data sebelum augmentasi: ", len(data)): Memberikan informasi jumlah data asli yang berhasil dimuat sebelum ditambah dengan data hasil augmentasi.

----

In [ ]:
def prepro3_visual(image):
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY) if len(image.shape) == 3 else image.copy()
    median_blur = cv.medianBlur(gray, 5)
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
    opening_result = cv.morphologyEx(median_blur, cv.MORPH_OPEN, kernel)
    return gray, median_blur, opening_result

dataPreprocessed = []

for i in range(len(data)):
    if data[i] is None: continue

    img_array = np.array(data[i], dtype=np.uint8)
    img_resized = cv.resize(img_array, (256, 256))
    
    _, _, final_prep = prepro3_visual(img_resized)
    dataPreprocessed.append(final_prep)

dataPreprocessed = np.array(dataPreprocessed)

print("Menampilkan Grafik Hasil Preprocessing (Sampel Acak):")
sample_idx = np.random.choice(len(dataPreprocessed), 3, replace=False)

for idx in sample_idx:
    img_array = np.array(data[idx], dtype=np.uint8)
    img_resized = cv.resize(img_array, (256, 256))
    img_rgb = cv.cvtColor(img_resized, cv.COLOR_BGR2RGB) if len(img_resized.shape) == 3 else img_resized
    
    gray, median, opening = prepro3_visual(img_resized)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img_rgb); axes[0].set_title("1. Asli (Resized)"); axes[0].axis('off')
    axes[1].imshow(gray, cmap='gray'); axes[1].set_title("2. Grayscale"); axes[1].axis('off')
    axes[2].imshow(median, cmap='gray'); axes[2].set_title("3. Median Filter"); axes[2].axis('off')
    axes[3].imshow(opening, cmap='gray'); axes[3].set_title("4. Morfologi Opening"); axes[3].axis('off')
    plt.tight_layout()
    plt.show()

print("Preprocessing untuk seluruh dataset selesai. Dimensi:", dataPreprocessed.shape)

In [ ]:
def glcm(image, derajat):
    angles = [0] if derajat == 0 else [np.pi/4] if derajat == 45 else [np.pi/2] if derajat == 90 else [3*np.pi/4]
    return graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)

def correlation(m): return graycoprops(m, 'correlation')[0, 0]
def dissimilarity(m): return graycoprops(m, 'dissimilarity')[0, 0]
def homogenity(m): return graycoprops(m, 'homogeneity')[0, 0]
def contrast(m): return graycoprops(m, 'contrast')[0, 0]
def ASM(m): return graycoprops(m, 'ASM')[0, 0]
def energy(m): return graycoprops(m, 'energy')[0, 0]
def entropyGlcm(m): return entropy(m.ravel())

In [ ]:
fitur = {k: [] for k in ['D0','D45','D90','D135','K0','K45','K90','K135','Dis0','Dis45','Dis90','Dis135',
                         'H0','H45','H90','H135','E0','E45','E90','E135','A0','A45','A90','A135',
                         'ER0','ER45','ER90','ER135','C0','C45','C90','C135']}

for img in dataPreprocessed:
    m0, m45, m90, m135 = glcm(img, 0), glcm(img, 45), glcm(img, 90), glcm(img, 135)
    
    for m, ang in zip([m0, m45, m90, m135], ['0', '45', '90', '135']):
        fitur[f'K{ang}'].append(contrast(m))
        fitur[f'Dis{ang}'].append(dissimilarity(m))
        fitur[f'H{ang}'].append(homogenity(m))
        fitur[f'E{ang}'].append(entropyGlcm(m))
        fitur[f'A{ang}'].append(ASM(m))
        fitur[f'ER{ang}'].append(energy(m))
        fitur[f'C{ang}'].append(correlation(m))

dataTable = {'Filename': file_name, 'Label': labels}
mapping = {'Contrast':'K', 'Dissimilarity':'Dis', 'Homogeneity':'H', 'Entropy':'E', 'ASM':'A', 'Energy':'ER', 'Correlation':'C'}
for name, prefix in mapping.items():
    for ang in ['0', '45', '90', '135']:
        dataTable[f'{name}{ang}'] = fitur[f'{prefix}{ang}']

df = pd.DataFrame(dataTable)
df.to_csv('hasil_ekstraksi_1.csv', index=False)

hasilEkstrak = pd.read_csv('hasil_ekstraksi_1.csv')
print("Tabel Hasil Ekstraksi Fitur GLCM (5 Baris Pertama):")
display(hasilEkstrak.head())

In [ ]:
correlation_matrix = hasilEkstrak.drop(columns=['Label','Filename']).corr()

threshold = 0.95 
columns = np.full((correlation_matrix.shape[0],), True, dtype=bool)

for i in range(correlation_matrix.shape[0]):
    for j in range(i+1, correlation_matrix.shape[0]):
        if abs(correlation_matrix.iloc[i,j]) >= threshold:
            if columns[j]: columns[j] = False

select = hasilEkstrak.drop(columns=['Label','Filename']).columns[columns]
x_new = hasilEkstrak[select]
y = hasilEkstrak['Label']

print("Grafik Heatmap Korelasi Antar Fitur (Yang Terseleksi):")
plt.figure(figsize=(14, 12))
sns.heatmap(x_new.corr(), annot=True, cmap='Blues', fmt=".2f")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x_new, y, test_size=0.2, random_state=42)

X_train_mean, X_train_std = X_train.mean(), X_train.std()
X_train = (X_train - X_train_mean) / X_train_std
X_test = (X_test - X_train_mean) / X_train_std

def generateClassificationReport(y_true, y_pred, title):
    print(f"------ {title} ------")
    print(classification_report(y_true, y_pred))
    print('Accuracy:', accuracy_score(y_true, y_pred), "\n")

rf = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(kernel='rbf', random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

models = {'Random Forest': rf, 'SVM': svm, 'KNN': knn}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_test = model.predict(X_test)
    generateClassificationReport(y_test, y_pred_test, f"{name} (Testing Set)")

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y_test))
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title(title)
    plt.show()

print("Grafik Confusion Matrix untuk Ketiga Model:")
plot_confusion_matrix(y_test, rf.predict(X_test), "Random Forest Confusion Matrix")
plot_confusion_matrix(y_test, svm.predict(X_test), "SVM Confusion Matrix")
plot_confusion_matrix(y_test, knn.predict(X_test), "KNN Confusion Matrix")

In [ ]:
dataPreprocessed = []

for i in range(len(data)):
    img_current = data[i]
    
    if img_current is None:
        continue
        
    # Langsung gunakan data[i] tanpa resize
    img_array = np.array(img_current, dtype=np.uint8)
    
    # Langsung masuk ke proses preprocessing
    gray, median_blur, opening_result = prepro3_visual(img_array)
    
    # ---------------------------------------------------------
    # VISUALISASI
    # ---------------------------------------------------------
    fig, axes = plt.subplots(1, 4, figsize=(9, 4))
    
    img_rgb = cv.cvtColor(img_array, cv.COLOR_BGR2RGB) if len(img_array.shape) == 3 else img_array
        
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"[{i}] Gambar Asli")
    axes[0].axis('off')
    
    axes[1].imshow(gray, cmap='gray')
    axes[1].set_title("1. Grayscale")
    axes[1].axis('off')
    
    axes[2].imshow(median_blur, cmap='gray')
    axes[2].set_title("2. Median Filter")
    axes[2].axis('off')
    
    axes[3].imshow(opening_result, cmap='gray')
    axes[3].set_title("3. Morfologi Opening")
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    dataPreprocessed.append(opening_result)

# PENTING: Jika ukuran gambar berbeda-beda, dataPreprocessed ini akan jadi 'list of arrays' 
# dan GLCM nanti mungkin akan error. 
dataPreprocessed = np.array(dataPreprocessed, dtype=object) 
print("Preprocessing selesai.")